# Step-by-Step Workflow

## Phase 1: Load & Explore
- Load `fear_greed_index.csv` (sentiment data)
- Load `historical_data.csv` (trading data)
- Check for missing values, duplicates, and data types

## Phase 2: Clean & Convert
- Convert date strings to datetime format (easier for analysis)
- Convert sentiment classes to categories (saves memory)
- Standardize column formatting (e.g., lowercase 'buy'/'sell')

## Phase 3: Merge
- Combine sentiment index with trading data (left join on date)
- Now each trade has a sentiment label

## Phase 4: Aggregate
- Group all trades by Account + Date
- Calculate daily metrics: total P&L, trade count, avg size, win rate, buy/sell ratio

## Phase 5: Export
- Save cleaned data to `final_dataset.csv`
- Ready for exploratory analysis!

---

In [2]:
# Import required libraries
import pandas as pd  # Data manipulation and analysis
import numpy as np  # Numerical operations (handles infinity, NaN values)
from datetime import datetime  # Convert Unix timestamps to readable dates
import warnings
warnings.filterwarnings('ignore')  # Suppress warning messages for clean output

In [3]:
# STEP 1: Load Fear & Greed Index Dataset
# This file contains daily sentiment scores (Fear/Greed Classification) for Bitcoin
df1 = pd.read_csv('fear_greed_index.csv')  # Load CSV into pandas DataFrame
df1.head()  # Display first 5 rows to verify data loaded correctly

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [4]:
# STEP 2: Load Historical Trading Dataset
# This file contains individual trade records from Hyperliquid (account, side, PnL, timestamp, etc.)
df2 = pd.read_csv('historical_data.csv')  # Load trade history CSV

In [5]:
df1.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [6]:
# Check for missing values in df1 (important for data quality)
# If a column shows >0, that many records have missing data for that column
df1.isna().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [7]:
# Check for completely duplicate rows in df1
# Duplicates can skew analysis, so we need to detect them
df1.duplicated().sum()

np.int64(0)

In [8]:
# Get statistical summary of numeric columns in df1

# Shows count, mean, std, min, max, quartiles
df1.describe()

,timestamp,value
count,2.644000e+03,2644.000000
mean,1.631899e+09,46.981089
std,6.597967e+07,21.827680
min,1.517463e+09,5.000000
25%,1.574811e+09,28.000000
50%,1.631900e+09,46.000000
75%,1.688989e+09,66.000000
max,1.746164e+09,95.000000


In [9]:
# Show data types and memory usage of all columns
# Helps identify if columns are correctly typed (int, float, string, etc.)
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   timestamp       2644 non-null   int64
 1   value           2644 non-null   int64
 2   classification  2644 non-null   str  
 3   date            2644 non-null   str  
dtypes: int64(2), str(2)
memory usage: 127.5 KB


In [10]:
# Count how many records fall into each sentiment classification

# Helps understand the distribution of Fear/Greed/Neutral days
df1['classification'].value_counts()

classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64

In [11]:
# STEP 3: Clean and Convert Data Types in df1
# Convert 'date' column to datetime format for proper date operations

df1['date'] = pd.to_datetime(df1['date'])# (Note: df2 date conversion is done separately below)

In [12]:
# Convert Unix timestamp (seconds since 1970) to human-readable datetime
# Example: 1693526400 becomes 2023-08-31 12:00:00
df1['timestamp'] = df1['timestamp'].apply(lambda x: datetime.fromtimestamp(x))

In [13]:
# Convert classification column to 'category' type for memory efficiency
# Categories like 'Fear', 'Greed' are repeated values, so this cuts memory usage
df1['classification'] = df1['classification'].astype('category')

In [14]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       2644 non-null   datetime64[us]
 1   value           2644 non-null   int64         
 2   classification  2644 non-null   category      
 3   date            2644 non-null   datetime64[us]
dtypes: category(1), datetime64[us](2), int64(1)
memory usage: 64.8 KB


In [15]:
# Show 6 random rows from df1 to spot-check the data
df1.sample(6)

,timestamp,value,classification,date
1596,2022-06-19 11:00:00,6,Extreme Fear,2022-06-19
1335,2021-10-01 11:00:00,27,Fear,2021-10-01
426,2019-04-06 11:00:00,65,Greed,2019-04-06
1005,2020-11-05 11:00:00,72,Greed,2020-11-05
113,2018-05-28 11:00:00,22,Extreme Fear,2018-05-28
352,2019-01-22 11:00:00,27,Fear,2019-01-22


In [16]:
# Show 4 random rows from df2 (historical trade data) to see structure
df2.sample(4)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
42888,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,HPOS,0.061400,1423.000,87.37,SELL,13-02-2024 15:04,60304.000,Close Long,25.549965,0xfe6ec2419b7ad6b35f0f0407995e8e015200da5ff732...,8719167746,False,-0.001747,7.590000e+14,1.710000e+12
86077,0xa0feb3725a9335f49874d7cd8eaad6be45b27416,HYPE,24.829000,0.650,16.14,BUY,15-12-2024 08:17,1967.470,Open Long,0.000000,0xff03f014cc33e20f0344041942b0f402022700b515c0...,55503395079,False,0.000806,5.910000e+14,1.730000e+12
35886,0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4,ZEREBRO,0.033082,348.000,11.51,BUY,20-02-2025 23:49,-24852.000,Close Short,0.270744,0xa87232f89d1e067e94cf041e2335b901a30087f78428...,73687447003,True,0.003867,2.940000e+14,1.740000e+12
206171,0x271b280974205ca63b716753467d5a371de622ab,PAXG,3082.900000,0.032,98.65,SELL,09-04-2025 18:49,-92.345,Open Short,0.000000,0xb65dd2a15afb0bec92bc042136250d0201b6003e7c6a...,85297612060,False,0.009470,7.780000e+14,1.740000e+12


In [17]:
# Check for duplicate rows in df2 (trading data should have unique trades)
df2.duplicated().sum()

np.int64(0)

In [18]:
# Check for missing values in df2 (critical: any trade with missing PnL is invalid)
df2.isna().sum()

Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [19]:
# STEP 4: Clean and Convert Data Types in df2
# Convert Timestamp IST (India Standard Time) to datetime format
# dayfirst=True tells pandas to interpret DD/MM/YYYY format (European style)
df2['Timestamp IST'] = pd.to_datetime(df2['Timestamp IST'],dayfirst=True)

In [20]:
# Extract just the DATE part from the full timestamp (ignore time)
# This creates a new 'date' column for matching with df1
df2['date'] = pd.to_datetime(df2['Timestamp IST'], dayfirst=True).dt.date
df2['date'] = pd.to_datetime(df2['date'])  # Convert back to datetime for consistency
# We only need date because we're aggregating trades by day

In [21]:
# Also convert the 'Timestamp' column (in case it's used later)
df2['Timestamp'] = pd.to_datetime(df2['Timestamp'], dayfirst=True)

In [22]:
# Count unique accounts in the dataset
# Tells us how many different traders we have data for
df2['Account'].value_counts().shape

(32,)

In [23]:
# Count unique transaction hashes (should match total rows if no duplicates)
df2['Transaction Hash'].value_counts().shape

(101184,)

In [24]:
# Check total rows and columns in df2
# Format: (number of rows, number of columns)
df2.shape

(211224, 17)

In [25]:
# STEP 5: Optimize Data Types in df2 for Memory & Performance
df2['Account'] = df2['Account'].astype('category')  # Categorical (repeated values: same accounts)
df2['Side'] = df2['Side'].str.lower()  # Standardize: convert all to lowercase
df2['Side'] = df2['Side'].astype('category')  # Categorical (only 'buy' or 'sell')
df2['Closed PnL'] = df2['Closed PnL'].astype('float32')  # Reduce memory: use 32-bit instead of 64-bit
df2['Size USD'] = df2['Size USD'].astype('float32')  # Same optimization for trade size

In [26]:
# Count unique cryptocurrencies traded in the dataset
df2['Coin'].unique().shape

(246,)

In [27]:
# Spot-check 6 random rows from cleaned df2
df2.sample(6)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date
73628,0xbaaaf6571ab7d571043ff1e313a9609a10637864,HYPE,13.4340,20.83,279.829987,buy,2025-03-11 05:52:00,29391.48,Open Long,0.000000,0x4e1a92a27b71c18487fc041f4ef1490201ce003d3a76...,78808028728,False,0.000000,5.160000e+14,1970-01-01 00:29:00,2025-03-11
18231,0x430f09841d65beb3f27765503d0f850b8bce7713,ZRO,5.2400,19.20,100.610001,sell,2024-07-23 03:12:00,64857.50,Close Long,36.091007,0x0bb7abaf213bce222f63040df08259012300f78dd94c...,30774951211,False,0.010060,8.380000e+14,1970-01-01 00:28:40,2024-07-23
28852,0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4,NTRN,1.4224,354.00,503.529999,sell,2024-02-14 08:22:00,7687.00,Close Long,16.461000,0xde2d293cde22daae160f04079c100001300079dd07f0...,9603646097,False,-0.010070,8.030000e+14,1970-01-01 00:28:30,2024-02-14
91727,0xa0feb3725a9335f49874d7cd8eaad6be45b27416,BERA,8.0547,43.60,351.179993,buy,2025-02-08 00:19:00,39.50,Open Long,0.000000,0x00000000000000000000000000000000000000000000...,70540589797,False,0.035118,3.610000e+14,1970-01-01 00:29:00,2025-02-08
43521,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,@4,2.3476,50.00,117.379997,buy,2024-06-13 21:27:00,4080.00,Buy,0.000000,0x7a3d80a44829a491d4e9040b6003910181005319d17e...,25980598070,True,0.017500,9.940000e+14,1970-01-01 00:28:40,2024-06-13
138793,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,LTC,132.9300,0.75,99.699997,sell,2025-01-30 19:46:00,0.00,Open Short,0.000000,0xf1e3ecba97379e46238f041ccf8ce302015100fc5785...,67759126133,True,0.034894,3.060000e+14,1970-01-01 00:29:00,2025-01-30


In [28]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Account           211224 non-null  category      
 1   Coin              211224 non-null  str           
 2   Execution Price   211224 non-null  float64       
 3   Size Tokens       211224 non-null  float64       
 4   Size USD          211224 non-null  float32       
 5   Side              211224 non-null  category      
 6   Timestamp IST     211224 non-null  datetime64[us]
 7   Start Position    211224 non-null  float64       
 8   Direction         211224 non-null  str           
 9   Closed PnL        211224 non-null  float32       
 10  Transaction Hash  211224 non-null  str           
 11  Order ID          211224 non-null  int64         
 12  Crossed           211224 non-null  bool          
 13  Fee               211224 non-null  float64       
 14  Trade ID       

In [29]:
# STEP 6: Merge Fear/Greed Index with Trading Data
# Join df1 (sentiment) with df2 (trades) on the 'date' column
# how='right' means: keep ALL rows from df2, add sentiment data where it exists
# Result: each trade row now has its corresponding sentiment label
df = pd.merge(df1, df2, on='date', how='right')

In [30]:
# Check a random row from merged dataset to verify it has both trade and sentiment data
df.sample()

,timestamp,value,classification,date,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
198635,2025-04-03 11:00:00,25.0,Fear,2025-04-03,0x4acb90e786d897ecffb614dc822eb231b4ffb9f4,SOL,117.44,1.74,204.350006,buy,2025-04-03 05:25:00,2094.32,Open Long,0.0,0xf583387cceb8fe00cf410420ca252402054200c43979...,82571786789,False,0.020434,5.710000e+14,1970-01-01 00:29:00


In [31]:
# Summary stats of merged dataset
df.describe()

,timestamp,value,date,Execution Price,Size Tokens,Size USD,Timestamp IST,Start Position,Closed PnL,Order ID,Fee,Trade ID,Timestamp
count,211218,211218.000000,211224,211224.000000,2.112240e+05,2.112240e+05,211224,2.112240e+05,211224.000000,2.112240e+05,211224.000000,2.112240e+05,211224
mean,2025-01-31 10:58:26.735221,51.649656,2025-01-30 23:54:28.674014,11414.723350,4.623365e+03,5.639451e+03,2025-01-31 12:04:22.915010,-2.994625e+04,48.749004,6.965388e+10,1.163967,5.628549e+14,1970-01-01 00:28:57.744290421
min,2023-05-01 11:00:00,10.000000,2023-05-01 00:00:00,0.000005,8.740000e-07,0.000000e+00,2023-05-01 01:06:00,-1.433463e+07,-117990.101562,1.732711e+08,-1.175712,0.000000e+00,1970-01-01 00:28:00
25%,2024-12-31 11:00:00,33.000000,2024-12-31 00:00:00,4.854700,2.940000e+00,1.937900e+02,2024-12-31 21:00:45,-3.762311e+02,0.000000,5.983853e+10,0.016121,2.810000e+14,1970-01-01 00:29:00
50%,2025-02-24 11:00:00,49.000000,2025-02-24 00:00:00,18.280000,3.200000e+01,5.970450e+02,2025-02-24 18:55:00,8.472793e+01,0.000000,7.442939e+10,0.089578,5.620000e+14,1970-01-01 00:29:00
75%,2025-04-02 11:00:00,72.000000,2025-04-02 00:00:00,101.580000,1.879025e+02,2.058960e+03,2025-04-02 18:22:00,9.337278e+03,5.792797,8.335543e+10,0.393811,8.460000e+14,1970-01-01 00:29:00
max,2025-05-01 11:00:00,94.000000,2025-05-01 00:00:00,109004.000000,1.582244e+07,3.921431e+06,2025-05-01 12:13:00,3.050948e+07,135329.093750,9.014923e+10,837.471593,1.130000e+15,1970-01-01 00:29:10
std,NaN,21.012784,NaN,29447.654868,1.042729e+05,3.657514e+04,NaN,6.738074e+05,919.164856,1.835753e+10,6.758854,3.257565e+14,NaN


In [32]:
# STEP 7: Aggregate Data by Account and Date
# Group all trades by (Account, Date) and calculate daily metrics
# This converts from trade-level data to daily-level data (one row per account per day)
t = df.groupby(['Account','date']).agg(
    daily_pnl = ('Closed PnL', 'sum'),          # SUM: total P&L for the day
    trade_count = ('Trade ID', 'count'),       # COUNT: how many trades that day
    avg_trade_size = ('Size USD','mean'),      # MEAN: average position size
    sentiment_value = ('value', 'first'),      # FIRST: sentiment score (same all day)
    sentiment_class = ('classification', 'first')  # FIRST: sentiment class (Greed/Fear/etc)
).reset_index()  # Convert grouped index back to regular columns

In [33]:
# STEP 8: Define Function to Calculate Buy/Sell Ratio
# This measures directional bias: ratio > 1 means more buys than sells (bullish)
def buy_sell_ratio(group):
    buy = (group['Side'] == 'buy').sum()   # Count buy trades
    sell = (group['Side'] == 'sell').sum()  # Count sell trades
    if(sell == 0):  # Avoid division by zero
        return np.inf  # Return infinity if only buys (extreme ratio)
    return buy / sell  # Return ratio (e.g., 2.0 = twice as many buys as sells)

In [34]:
# STEP 9: Define Function to Calculate Win Rate
# This measures % of profitable trades (PnL > 0)
def win_rate(group):
    win = (group['Closed PnL'] > 0).sum()  # Count trades with positive P&L
    total = (group['Closed PnL']).count()  # Count total trades
    
    if total == 0:  # Avoid division by zero
        return 0  # No trades = 0% win rate
    return win / total  # Return percentage (e.g., 0.45 = 45% win rate)

In [35]:
# STEP 10: Calculate Win Rate for Each Account-Date Combination
# Apply the win_rate function to each group and add as new column
t['win_rate'] = df.groupby(['Account','date']).apply(win_rate).values

In [36]:
# STEP 11: Calculate Buy/Sell Ratio for Each Account-Date Combination
# Apply the buy_sell_ratio function and add as new column
t['buy_sell_ratio'] = df.groupby(['Account','date']).apply(buy_sell_ratio).values

In [38]:
# STEP 12: Export Final Dataset
# Save the cleaned and engineered data to CSV for use in EDA notebook
# This file is ready for exploratory analysis and visualization
t.to_csv('final_dataset.csv', index=False)  # index=False means don't save row numbers

---

##  Summary: What This Notebook Does

### Overview
This notebook transforms raw trading data into a clean, analysis-ready dataset by combining:
1. **Fear & Greed Index** (sentiment labels for each day)
2. **Trading Data** (individual trade records with timestamps, accounts, P&L, etc.)

### The 12 Steps:

| # | Step | Input | Output | Purpose |
|---|------|-------|---------|---------|
| 1-2 | Load Data | CSV files | DataFrames (df1, df2) | Load source data into memory |
| 3-5 | Explore df1 | df1 | Printed statistics | Understand data structure, check for missing values, duplicates |
| 6 | Clean Types (df1) | df1 | Converted columns | Fix date formats, optimize memory with categories |
| 7-9 | Explore df2 | df2 | Printed statistics | Understand trading data structure |
| 10-11 | Clean Types (df2) | df2 | Converted columns | Standardize formats, optimize memory |
| 12 | Merge | df1 + df2 | df | Combine sentiment with trades |
| 13 | Aggregate | df | t (daily records) | Group trades by account & date → one row per day |
| 14-15 | Feature Engineering | t | t with new columns | Calculate win_rate and buy_sell_ratio |
| 16 | Export | t | final_dataset.csv | Save for EDA analysis |

### Input Files (Required)
```
├── fear_greed_index.csv
│   ├── date: Date of sentiment reading
│   ├── value: Numeric sentiment score (0-100)
│   ├── timestamp: Unix timestamp
│   └── classification: Sentiment class (Extreme Fear, Fear, Neutral, Greed, Extreme Greed)
│
└── historical_data.csv
    ├── Account: Trader's wallet address
    ├── Timestamp IST: When the trade occurred
    ├── Side: 'Buy' or 'Sell'
    ├── Coin: Asset traded (e.g., BTC, ETH)
    ├── Size USD: Trade size in USD
    ├── Closed PnL: Trade profit/loss
    ├── Trade ID: Unique trade identifier
    └── Transaction Hash: Blockchain transaction ID
```

### Output File: `final_dataset.csv`
```
| Account | date | daily_pnl | trade_count | avg_trade_size | sentiment_value | sentiment_class | win_rate | buy_sell_ratio |
